# Generative AI 013 — Text Splitters

No LangChain splitter package is installed in this course's environment, which
turns out to suit the topic: a splitter is a small algorithm, so we implement
the two that matter and **measure** what they do differently.

| Part | What we check |
|---|---|
| A | length-based splitting cuts **2 of 3** boundaries inside a word |
| B | `chunk_overlap` costs exactly **1/(1 − overlap)** |
| C | the recursive splitter, in ~15 lines, at three chunk sizes |
| D | same chunks, same limit, **0** broken words |
| E | semantic splitting — the mechanism, and where it collapses |

Needs `scikit-learn` for Part E; the rest is plain Python.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np

SAMPLE = (
    "Cricket is a bat-and-ball game played between two teams of eleven players. "
    "The field is oval and at its centre is a rectangular pitch twenty-two yards long. "
    "Each team takes turns to bat and to bowl, and the side with more runs wins. "
    "Test cricket can last five days and still end in a draw, which surprises newcomers."
)
print(len(SAMPLE), "characters")

## Part A — Length-based splitting

Pick a size, walk the text, cut every N characters. Four lines, very fast.

In [ ]:
def char_split(text, chunk_size, chunk_overlap=0):
    step = chunk_size - chunk_overlap
    return [text[i:i + chunk_size] for i in range(0, len(text), step)]

chunks = char_split(SAMPLE, 100)
for c in chunks[:3]:
    print(repr(c))

In [ ]:
# Where did the cuts actually land?
cuts = [SAMPLE[i*100-1:i*100+1] for i in range(1, len(chunks))]
mid = sum(1 for c in cuts if c[0].isalnum() and c[1].isalnum())
print(f"{mid} of {len(cuts)} cuts fall INSIDE a word:", cuts)

assert mid == 2
print()
print("One chunk ends '...to bat and to bowl, a' and the next begins")
print("'nd the side with more runs...'. Embed that and you get a vector for")
print("nothing in particular - and a chunk is the unit a retriever hands to")
print("a model, so this is a citation that starts mid-thought.")

## Part B — What `chunk_overlap` costs

In [ ]:
long_text = SAMPLE * 8
print(f"a {len(long_text)}-character document, chunk_size=100\n")
print(f"{'overlap':>9}{'chunks':>9}{'vs none':>10}")
base = len(char_split(long_text, 100, 0))
rows = []
for ov in (0, 10, 20, 50, 80):
    n = len(char_split(long_text, 100, ov))
    rows.append((ov, n))
    print(f"{ov:>8}%{n:>9}{n/base:>9.2f}x")

# The cost is exactly 1/(1 - overlap fraction).
for ov, n in rows[1:]:
    predicted = base / (1 - ov/100)
    assert abs(n - predicted) / predicted < 0.10, (ov, n, predicted)
print("\nmatches 1/(1 - overlap) within 10% at every setting")

The window advances by `chunk_size − overlap`, so halving the step doubles the
chunks. Every extra chunk is another vector to store, embed and search.

**10–20% costs about 1.2×. 80% costs about 4.9×** for the same document.

## Part C — The recursive splitter

In [ ]:
def recursive_split(text, chunk_size, separators=None):
    """Try the biggest separator first; recurse only when a piece is too big."""
    if separators is None:
        separators = ["\n\n", "\n", " ", ""]     # paragraph, line, word, char

    def split(t, seps):
        if len(t) <= chunk_size or not seps:
            return [t]
        sep, rest = seps[0], seps[1:]
        pieces = list(t) if sep == "" else t.split(sep)
        out = []
        for p in pieces:
            out.extend([p] if len(p) <= chunk_size else split(p, rest))
        return out

    merged, current = [], ""
    for p in split(text, separators):
        candidate = f"{current} {p}".strip() if current else p
        if len(candidate) <= chunk_size:
            current = candidate
        else:
            if current:
                merged.append(current)
            current = p
    if current:
        merged.append(current)
    return merged

WORKED = "My name is Nitish\nI am 35 years old\n\nI live in Gurgaon\nHow are you"
for size in (10, 25, 50):
    out = recursive_split(WORKED, size)
    print(f"chunk_size={size:<3} -> {len(out)} chunks")
    for c in out:
        print(f"      {c!r}")
    print()

assert len(recursive_split(WORKED, 10)) == 8     # word level
assert len(recursive_split(WORKED, 25)) == 4     # sentence level
assert len(recursive_split(WORKED, 50)) == 2     # paragraph level

Raise the limit and it splits at a **higher** level, because fewer pieces are
too big to keep whole. `chunk_size` is really a choice of which natural
boundary to cut on.

> *Fidelity note:* this toy version re-joins merged pieces with a space, so a
> newline inside a chunk becomes `" "`. LangChain re-joins with the separator
> it split on. Which **level** it splits at — the thing being taught — is the
> same.

## Part D — The two, side by side

In [ ]:
def starts_mid_word(chunks, text):
    n = 0
    for c in chunks:
        i = text.find(c)
        if i > 0 and text[i-1].isalnum() and c[:1].isalnum():
            n += 1
    return n

char_chunks = char_split(SAMPLE, 100)
rec_chunks  = recursive_split(SAMPLE, 100)

print(f"{'splitter':<12}{'chunks':>8}{'longest':>9}{'mid-word starts':>18}")
for name, ch in (("character", char_chunks), ("recursive", rec_chunks)):
    print(f"{name:<12}{len(ch):>8}{max(len(c) for c in ch):>9}"
          f"{starts_mid_word(ch, SAMPLE):>18}")

assert starts_mid_word(char_chunks, SAMPLE) == 2
assert starts_mid_word(rec_chunks, SAMPLE) == 0
assert max(len(c) for c in rec_chunks) <= 100      # limit still respected
print("\nSame count, limit respected by both, and no broken words.")
print("There is no trade-off here - it simply chose better places to cut.")

## Part E — Semantic splitting

Structure fails when one paragraph holds two subjects. There is no separator
to split on, so only **meaning** can find the boundary.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def neighbour_sims(text):
    sents = [x.strip() + "." for x in text.split(". ") if x.strip()]
    V = TfidfVectorizer(stop_words="english").fit_transform(sents).toarray()
    return [float(cosine_similarity([V[i]], [V[i+1]])[0][0])
            for i in range(len(sents)-1)]

MIXED = ("The monsoon decides Indian agriculture every single year. "
         "A weak monsoon means farmers lose the harvest they depend on. "
         "Farmers therefore watch the monsoon forecast more closely than any market. "
         "The IPL auction fills a crowded room every winter. "
         "Franchises bid against each other at the auction for the players they want. "
         "One good auction can decide which players carry a team all season.")

sims = neighbour_sims(MIXED)
for i, x in enumerate(sims):
    mark = "   <- topic changes" if x == min(sims) else ""
    print(f"sentence {i} -> {i+1}: {x:.4f}{mark}")

assert int(np.argmin(sims)) == 2      # agriculture becomes cricket

In [ ]:
# Now break it. The same two topics, written WITHOUT repeating key terms.
SPARSE = ("Indian agriculture depends heavily on the monsoon. "
          "Rainfall in June and July decides the kharif harvest. "
          "Farmers watch the forecast closely every year. "
          "The IPL auction happens in the winter months. "
          "Franchises bid for players in a crowded room. "
          "A good auction can decide a team's whole season.")

sparse_sims = neighbour_sims(SPARSE)
print(sparse_sims)

assert all(x == 0.0 for x in sparse_sims)
print()
print("Every pair EXACTLY zero. No minimum, so no boundary to find.")
print("Six short sentences on two obvious topics, and a lexical method sees")
print("six unrelated strings - because consecutive sentences rarely reuse a")
print("content word unless the writer happens to.")
print()
print("Same failure as lesson 003's paraphrase. The first table shows the")
print("MECHANISM; this one shows how narrow the conditions were. Real")
print("semantic chunking needs a real embedding model.")

## What to take away

- Length-based splitting cut **2 of 3** boundaries inside a word.
- `chunk_overlap` costs exactly **1/(1 − overlap)** — 20% is 1.23×, 80% is 4.88×.
- The recursive splitter gave the **same chunk count** with **0** broken words.
- `chunk_size` chooses which boundary it cuts on: word, sentence, paragraph.
- Semantic splitting found the boundary as a single **0.0000** — and returned
  **all zeros** when the topics stopped repeating their vocabulary.

## Exercises

1. Add `chunk_overlap` to `recursive_split`. Where does the overlap have to
   come from — characters, or whole pieces?
2. Give the recursive splitter code separators (`"\nclass "`, `"\ndef "`) and
   split a Python file. Does a class land in its own chunk?
3. Part A counted cuts inside words. Count cuts inside *sentences* instead.
   Which metric would you rather optimise, and why?
4. Find the chunk_size at which the worked example switches from sentence to
   paragraph level. Predict it from the string lengths before running it.
5. Part E broke on sparse vocabulary. Write a third version of the two topics
   where TF-IDF finds the *wrong* boundary — not no boundary.